#### **Initialization**

In [1]:
# CITE THE LIBRARYYYYYYYYYYYYYYYYYY
import openai
import os
import json

# Security Measure
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

openai.api_key  = os.getenv('OPENAI_API_KEY')

##### Load the KG:

In [2]:
import rdflib

eu_graph = rdflib.Dataset().parse("eu_legislation_graph.nq", format="nquads")

#### Prepare the Agent class:

In [3]:
import openai
import json
import time

class Legal_Agent:
    def __init__(self, system_prompt="", step_prompts=[], api_key="", tool_table=dict()):
        self.system_prompt = system_prompt
        self.step_prompts = step_prompts
        self.context = [{"role": "system", "content": system_prompt}]
        self.reasoning_log = []
        self.tool_lookup = {value['function']['name']: func for func, value in tool_table.items()}
        self.tools = [value for value in tool_table.values()]
        self.instance = openai.OpenAI(
            base_url="https://integrate.api.nvidia.com/v1",
            api_key=api_key
        )
        print(f"[INIT] Legal_Agent initialized.")
        print(f"[INIT] Tools registered: {list(self.tool_lookup.keys())}")
        print(f"[INIT] Step prompts loaded: {len(self.step_prompts)}")

    # -------------------------------------------------------------------------
    # Core response method (non-streaming)
    # -------------------------------------------------------------------------
    def respond_to(self, query="", role="user", _depth=0):
        indent = "  " * _depth  # Indent nested (tool-loop) calls for readability

        if query:
            print(f"{indent}[RESPOND] Appending {role} message ({len(query)} chars)")
            self.context.append({"role": role, "content": query})

        print(f"{indent}[RESPOND] Sending request to model "
              f"(context length: {len(self.context)} messages)...")
        t0 = time.perf_counter()

        answer = self.instance.chat.completions.create(
            model="openai/gpt-oss-120b",
            reasoning_effort="high",
            messages=self.context,
            temperature=0,
            top_p=0.1,
            tools=self.tools,
            max_tokens=None,
            stream=False
        )

        elapsed = time.perf_counter() - t0
        print(f"{indent}[RESPOND] Model responded in {elapsed:.2f}s | "
              f"finish_reason={answer.choices[0].finish_reason} | "
              f"usage={answer.usage}")

        reasoning = getattr(answer.choices[0].message, "reasoning_content", None)
        if reasoning:
            print(f"{indent}[REASONING] {str(reasoning)[:200]}{'...' if len(str(reasoning)) > 200 else ''}")
        self.reasoning_log.append(str(reasoning))

        # --- Tool call branch ---
        if answer.choices[0].message.tool_calls:
            tool_call  = answer.choices[0].message.tool_calls[0]
            tool_name  = tool_call.function.name
            args       = json.loads(tool_call.function.arguments)

            print(f"{indent}[TOOL] Calling '{tool_name}' with args: {args}")

            self.context.append(answer.choices[0].message)

            if tool_name not in self.tool_lookup:
                raise KeyError(f"[TOOL] Unknown tool requested by model: '{tool_name}'")

            t1 = time.perf_counter()
            tool_result = self.tool_lookup[tool_name](**args)
            tool_elapsed = time.perf_counter() - t1

            result_preview = str(tool_result)[:300]
            print(f"{indent}[TOOL] '{tool_name}' returned in {tool_elapsed:.2f}s: "
                  f"{result_preview}{'...' if len(str(tool_result)) > 300 else ''}")

            self.context.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": tool_name,
                "content": str(tool_result)
            })

            print(f"{indent}[TOOL] Re-entering respond_to (depth {_depth + 1})...")
            return self.respond_to(_depth=_depth + 1)

        # --- Normal response branch ---
        content = answer.choices[0].message.content
        print(f"{indent}[RESPOND] Final answer ({len(content)} chars): "
              f"{content[:150]}{'...' if len(content) > 150 else ''}")

        self.context.append({"role": "assistant", "content": content})
        return content

    # -------------------------------------------------------------------------
    # Sequential / conditional reasoning (unchanged logic, traces added)
    # -------------------------------------------------------------------------
    def sequential_reasoning(self, query="", max_depth=10):
        print(f"[SEQ] Starting sequential_reasoning | max_depth={max_depth}")
        print(f"[SEQ] Query: {query[:200]}{'...' if len(query) > 200 else ''}")
        state = {"query": query}
        return self.conditional_call(
            step_index=0,
            depth=0,
            max_depth=max_depth,
            state=state
        )

    def conditional_call(self, step_index, depth, max_depth, state):
        print(f"[COND] Step {step_index + 1} | depth={depth}/{max_depth}")

        if depth > max_depth:
            raise RuntimeError(
                f"[COND] Max reasoning depth ({max_depth}) reached at step {step_index + 1}"
            )

        if step_index >= len(self.step_prompts):
            raise IndexError(
                f"[COND] step_index {step_index} out of range "
                f"(only {len(self.step_prompts)} step prompts defined)"
            )

        prompt = self.step_prompts[step_index].format(**state)
        print(f"[COND] Formatted prompt ({len(prompt)} chars): "
              f"{prompt[:150]}{'...' if len(prompt) > 150 else ''}")

        raw_result = self.respond_to(prompt, "user")

        try:
            state = json.loads(raw_result)
        except json.JSONDecodeError as e:
            print(f"[COND] ⚠ JSON parse failed: {e}")
            print(f"[COND] Raw result was: {raw_result[:300]}")
            raise

        next_step = state.get("next_step", None)
        print(f"[COND] Step {step_index + 1} complete | next_step={next_step} | "
              f"state keys={list(state.keys())}")

        if next_step is None:
            print(f"[COND] No next_step — sequential reasoning complete.")
            return state

        return self.conditional_call(
            step_index=(next_step - 1),
            depth=(depth + 1),
            max_depth=max_depth,
            state=state
        )

    # -------------------------------------------------------------------------
    # Utility methods
    # -------------------------------------------------------------------------
    def reset_working_memory(self):
        print("[RESET] Clearing context and reasoning log (full reset).")
        self.context = [{"role": "system", "content": self.system_prompt}]
        self.reasoning_log = []

    def reset_context(self):
        print("[RESET] Clearing context only (reasoning log preserved).")
        self.context = [{"role": "system", "content": self.system_prompt}]

    def export_context(self):
        print(f"[EXPORT] Exporting context ({len(self.context)} messages).")
        return self.context

#### Prompts:

In [ ]:
# Cite the SOURCEEEEEEEEEEEEEEE!!!!!!!!!!!!!!!!!!!!!!!!!
practice_areas = [
    "Bankruptcy and insolvency law",
    "Commercial law",
    "Consumer law",
    "Criminal law",
    "Employment law",
    "EU law",
    "Family law",
    "Human rights and civil liberties",
    "Immigration and asylum law",
    "Intellectual property",
    "Information Technology (IT) law",
    "Litigation, mediation, arbitration",
    "Personal injury, damage to goods",
    "Property law",
    "Public law",
    "Social security law",
    "Succession law",
    "Tax law",
    "Traffic and transport law"
]

# EQUIP THE RECEPTIONIST WITH A QUERY TOOL?
receptionist_prompt = f"""
You are a front-facing receptionist that handles inquiries about EU law.
You are required to do the following:

1.  Reformulate and improve queries;
2.  Identify ALL practice areas RELEVANT to the reformulated query;
3.  Respond to queries accordingly.

You will be prompted to do each of these tasks, one at a time.
"""

receptionist_step_prompts = [
    """
    1.  Reformulate the user's query via decomposition and/or adding sufficient context.
        Ensure that queries/sub-queries are atomic to facilitate downstream SPARQL queries.
        Collect the resulting query/sub-queries in a list.
        Determine if the user's query is general or complex.
        If the query is general, proceed to step 3 after the outputting the JSON dictionary.

        Output the following JSON dictionary:
        {{
            "improved_query": [reformulated_query_list],
            "next_step": 2 or 3 # depending on step 1
        }}

        User query: {query}
    """,

    """
    2.  List ALL practice areas from the list below RELEVANT to the reformulated query:
        Bankruptcy and insolvency law, Commercial law, Consumer law, Criminal law, Employment law, EU law, Family law, Human rights and civil liberties, Immigration and asylum law, Intellectual property, Information Technology (IT) law, Litigation, mediation, arbitration, Personal injury, damage to goods, Property law, Public law, Social security law, Succession law, Tax law, Traffic and transport law

        Output the following JSON dictionary:
        {{
            "improved_query": [reformulated_query_list],
            "practice_areas": ["area_1", "area_2", "area_3", ...],
            "next_step": 3
        }}

        Reformulated Query: {improved_query}
    """,

    """
    3.  Respond to the reformulated query.
        If you performed step 2, inform the user that you shall forward the query all relevant experts in the identified practice areas instead.

        Output the following JSON dictionary:
        {{
            "improved_query": [reformulated_query_list],
            "practice_areas": ["area_1", "area_2", "area_3", ...],
            "response": "your_response"
        }}

        Reformulated query: {improved_query}
    """
]

sparql_agent_prompt = """
You are a sub-agent that performs SPARQL query operations.
You are required to do the following:

1.  Choose an appropriate query template that best matches the user query;
    Extract synonyms from keywords in the provided user query.
2.  Construct a statement for each quad (source document + triple) returned from the results.

You will be prompted to do each of these tasks, one at a time.
DO NOT answer the user directly or interpret results beyond verbalization.
"""

# CURRENTLY EXCLUDES 1-HOP QUERIES
sparql_step_prompts = [
    """
    1.  DETERMINE ONE query type from the list below that matches the logic of the original query:
        ["and", "or", "not", "and_or_not", "subject_only", "predicate_only", "object_only", "subject_and_object", "subject_and_predicate", "predicate_and_object", "subject_or_object", "subject_or_predicate", "predicate_or_object"]
    
        Then, for each component required in the query, i.e. subject, predicate, object, determine the most relevant synonyms.
        For example, the query "What information is there on cars?" belongs to the "subject_or_object" type, so synonyms for either the subject or object are needed.

        When compiling synonyms, prefer short word stems and partial words over full phrases, and break up multi-word phrases into their individual keywords.
        For example, prefer "climat" over "climate policy", and "polic" over "policy" to capture both "policy" and "policies" in a single term.
        For multi-word concepts, break the phrase into its component stems and include each as a separate synonym rather than keeping the phrase whole —
        for example, represent "climate policy" as separate stems "climat" and "polic" rather than the full phrase, so that CONTAINS can match each root independently across morphological variants and compound terms.
        This makes CONTAINS matching more robust by catching morphological variants and compound terms that share a common root.

        Call the sparql_query tool and pass in the query type and the compiled synonyms for each triple component.

    Output the following JSON dictionary:
        {{
            "results": ["triples_returned_from_the_sparql_tool"], 
            "next_step": 2
        }}

        User query: {query}
    """,

    """
    2.  Construct a statement for each quad (source, subject, predicate, object) returned from the results in the following format:
        "Source: Statement in natural language"
    
    Output the following JSON dictionary:
        {{
            "results": ["triples_returned_from_the_sparql_tool"],
            "statements": ["list_of_source-labeled_statements"]
        }}

        Results: {results}

    """
    ]

legal_expert_list = []

for area in practice_areas:
    prompt = f"""
    You are a EU legal researcher in the following practice area: {area}.
    You are required to analyze the given queries/sub-queries STRICTLY WITHIN your practice area.
    You will be tasked to do the following:

    1.  For each query/sub-query, call a SPARQL sub-agent via the call_sparql_subagents tool.
    2.  Use the IRAC (Issue-Rules-Application-Conclusion) framework to analyze the results.
    3.  Organize the results into a coherent argument.

    You will be prompted to do each of these tasks, one at a time.
    """

    step_prompts = [
        """
        1.  For each query/sub-query, call a SPARQL sub-agent via the call_sparql_subagents tool.
            ONLY SEARCH FOR INFORMATION WITHIN YOUR PRACTICE AREA.

            Output the following JSON dictionary:
            {{
                "compiled_results": [list of passed statements/propositions],
                "queries_to_recall": [keep this blank, it's for Step 2],
                "next_step": 2
            }}

            Query List: {query}
        """,

        """
        2.  Evaluate the results returned by the SPARQL sub-agent(s) using the IRAC (Issue-Rules-Application-Conclusion) framework.
            If the results are satisfactory, move on to step 3;
            Otherwise, if the results are inadequate or incomprehensive,
                reformulate the corresponding query/subquery and call a SPARQL sub-agent via the sparql_agent tool.

            Output the following JSON dictionary:
            {{
                "compiled_results": [list of passed statements/propositions],
                "queries_to_recall": [list of reformulated queries],
                "next_step": 2 or 3 # depending on the results
            }}

            Compiled Results: {compiled_results}
            Query(ies) to Recall: {queries_to_recall}
        """,

        """
        3.  Organize the results into a coherent argument, as a list of propositions.
            LIST OUT ALL PROPOSITIONS, including refuting ones.

            Output the following JSON dictionary:
            {{
                "practice_area": "your practice area",
                "supporting_propositions": [list of supporting statements/propositions],
                "refuting_propositions": [list of refuting statements/propositions]
            }}

            "Compiled Results: {compiled_results}"
        """
    ]

    legal_expert_list.append({"prompt": prompt, "step_prompts": step_prompts, "practice_area": area})

expert_api_keys = [openai.api_key for expert in legal_expert_list]
for index, api in enumerate(expert_api_keys):
    legal_expert_list[index]["api_key"] = api

evaluator_prompt = """
You are a judge with comprehensive knowledge of EU law, across all but not limited to the following practice areas:
Bankruptcy and insolvency law, Commercial law, Consumer law, Criminal law, Employment law, EU law, Family law, Human rights and civil liberties, Immigration and asylum law, Intellectual property, Information Technology (IT) law, Litigation, mediation, arbitration, Personal injury, damage to goods, Property law, Public law, Social security law, Succession law, Tax law, Traffic and transport law

You are required to do the following:
1.  Generate a COMPREHENSIVE EVALUATION on the compiled analyses through the IRAC (Issue-Rules-Application-Conclusion) framework.
2.  Compile supporting and refuting statements/propositions.
2.  Spot inconsistencies or conflicting rules and highlight them.
3.  Provide recommendations/strategies based on your analysis, and predict the most likely outcome(s), if applicable.

Output Format:
{{
    "evaluation": "a comprehensive evaluation",
    "supporting_props": [list of supporting statements/propositions],
    "refuting_props": [list of refuting statements/propositions],
    "inconsistencies": [list of conflicting/inconsistent rules],
    "recs_and strats": [list of recommendations and strategies],
    "likely_outcomes": [most likely outcomes, if applicable]
}}

Compiled Analyses: {query}
"""

#### Tools:

In [12]:
from concurrent.futures import ThreadPoolExecutor

import re

def sparql_query(
    synonyms: dict[str, list[str]],
    query_type: str,
    ) -> list[tuple[str, str, str, str]]:
    """
    Builds a parameterized SPARQL query from a synonym dictionary and returns the query results as a list of (subject, predicate, object, source) quads.

    Parameters
    ----------
    synonyms : dict
        Keys: "subject", "predicate", "object" — each maps to a list of synonyms.
        Example:
            {
                "subject":   ["person", "individual"],
                "predicate": ["knows", "friendOf"],
                "object":    ["city", "town"],
            }

    query_type : str
        One of:
            Logical  : "and" | "or" | "not" | "and_or_not"
            Partial  : "subject_only" | "predicate_only" | "object_only"
                       | "subject_and_object" | "subject_and_predicate" | "predicate_and_object"
                       | "subject_or_object"  | "subject_or_predicate"  | "predicate_or_object"
            One-hop  : "1hop_fixed" | "1hop_variable" | "1hop_optional"

    Returns
    -------
    list[tuple[str, str, str, str]]
        Each tuple is (source, subject, predicate, object).
    """

    # ------------------------------------------------------------------
    # 1. Validate inputs
    # ------------------------------------------------------------------
    print(f"[SPARQL] Called with query_type='{query_type}' | "
          f"synonym keys={list(synonyms.keys())} | "
          f"synonym counts={ {k: len(v) for k, v in synonyms.items()} }")

    REQUIRED_KEYS_BY_TYPE = {
        "and":                    {"subject", "predicate", "object"},
        "or":                     {"subject", "predicate", "object"},
        "not":                    {"subject", "predicate", "object"},
        "and_or_not":             {"subject", "predicate", "object"},
        "subject_only":           {"subject"},
        "predicate_only":         {"predicate"},
        "object_only":            {"object"},
        "subject_and_object":     {"subject", "object"},
        "subject_and_predicate":  {"subject", "predicate"},
        "predicate_and_object":   {"predicate", "object"},
        "subject_or_object":      {"subject", "object"},
        "subject_or_predicate":   {"subject", "predicate"},
        "predicate_or_object":    {"predicate", "object"},
        "1hop_fixed":             {"subject", "predicate", "object"},
        "1hop_variable":          {"subject", "predicate", "object"},
        "1hop_optional":          {"subject", "predicate", "object"},
    }

    qt = query_type.strip().lower()

    if qt not in REQUIRED_KEYS_BY_TYPE:
        print(f"[SPARQL] ✗ Unknown query_type='{qt}'")
        raise ValueError(f"Unknown query_type '{query_type}'. Valid: {sorted(REQUIRED_KEYS_BY_TYPE)}")

    required_keys = REQUIRED_KEYS_BY_TYPE[qt]
    print(f"[SPARQL] Required synonym keys for '{qt}': {required_keys}")

    missing = required_keys - synonyms.keys()
    if missing:
        print(f"[SPARQL] ✗ Missing synonym keys: {missing}")
        raise ValueError(f"query_type='{qt}' requires synonym keys: {missing}")

    for key in required_keys:
        if not isinstance(synonyms[key], list) or not synonyms[key]:
            print(f"[SPARQL] ✗ synonyms['{key}'] is empty or not a list: {synonyms.get(key)}")
            raise ValueError(f"synonyms['{key}'] must be a non-empty list (required by query_type='{qt}')")

    print(f"[SPARQL] ✓ Validation passed")

    # ------------------------------------------------------------------
    # 2. Helper: build a CONTAINS+LCASE filter for a list of synonyms
    # ------------------------------------------------------------------
    def contains_filter(var: str, terms: list[str]) -> str:
        clauses = [
            f'CONTAINS(LCASE(str(?{var})), "{t.lower()}")'  # re.escape() removed
            for t in terms
        ]
        return "(" + " || ".join(clauses) + ")"

    s_f = contains_filter("subject",   synonyms["subject"])   if "subject"   in required_keys else None
    p_f = contains_filter("predicate", synonyms["predicate"]) if "predicate" in required_keys else None
    o_f = contains_filter("object",    synonyms["object"])    if "object"    in required_keys else None

    print(f"[SPARQL] Filters built | s_f={'yes' if s_f else 'skipped'} "
          f"p_f={'yes' if p_f else 'skipped'} o_f={'yes' if o_f else 'skipped'}")

    # ------------------------------------------------------------------
    # 3. Build the SPARQL query string
    # ------------------------------------------------------------------
    if qt == "and":
        query = f"""
        SELECT ?graph ?subject ?predicate ?object WHERE {{
            GRAPH ?graph {{
                ?subject ?predicate ?object .
                FILTER({s_f})
                FILTER({p_f})
                FILTER({o_f})  
            }}    
        }}
        """

    elif qt == "or":
        query = f"""
        SELECT ?graph ?subject ?predicate ?object WHERE {{
            GRAPH ?graph {{
                ?subject ?predicate ?object .
                FILTER({s_f} || {p_f} || {o_f})
            }}
        }}
        """

    elif qt == "not":
        query = f"""
        SELECT ?graph ?subject ?predicate ?object WHERE {{
            GRAPH ?graph {{
                ?subject ?predicate ?object .
                FILTER({s_f})
                FILTER({p_f})
                FILTER(!{o_f})
            }}
        }}
        """

    elif qt == "and_or_not":
        mid = max(1, len(synonyms["object"]) // 2)
        obj_a = contains_filter("object", synonyms["object"][:mid])
        obj_b = contains_filter("object", synonyms["object"][mid:] or synonyms["object"])
        query = f"""
        SELECT ?graph ?subject ?predicate ?object WHERE {{
            GRAPH ?graph {{
                ?subject ?predicate ?object .
                FILTER(
                    {s_f}
                    && ({obj_a} || {obj_b})
                    && !{p_f}
                )
            }}
        }}
        """

    elif qt == "subject_only":
        query = f"""
        SELECT ?graph ?subject ?predicate ?object WHERE {{
            GRAPH ?graph {{
                ?subject ?predicate ?object .
                FILTER({s_f})
            }}
        }}"""

    elif qt == "predicate_only":
        query = f"""
        SELECT ?graph ?subject ?predicate ?object WHERE {{
            GRAPH ?graph {{
                ?subject ?predicate ?object .
                FILTER({p_f})
            }}
        }}
        """

    elif qt == "object_only":
        query = f"""
        SELECT ?graph ?subject ?predicate ?object WHERE {{
            GRAPH ?graph {{
                ?subject ?predicate ?object .
                FILTER({o_f})
            }}
        }}
        """

    elif qt == "subject_and_object":
        query = f"""
        SELECT ?graph ?subject ?predicate ?object WHERE {{
            GRAPH ?graph {{
                ?subject ?predicate ?object .
                FILTER({s_f})
                FILTER({o_f})
            }}
        }}
        """

    elif qt == "subject_and_predicate":
        query = f"""
        SELECT ?graph ?subject ?predicate ?object WHERE {{
            GRAPH ?graph {{
                ?subject ?predicate ?object .
                FILTER({s_f})
                FILTER({p_f})
            }}
        }}
        """

    elif qt == "predicate_and_object":
        query = f"""
        SELECT ?graph ?subject ?predicate ?object WHERE {{
            GRAPH ?graph {{
                ?subject ?predicate ?object .
                FILTER({p_f})
                FILTER({o_f})
            }}
        }}
        """

    elif qt == "subject_or_object":
        query = f"""
        SELECT ?graph ?subject ?predicate ?object WHERE {{
            GRAPH ?graph {{
                ?subject ?predicate ?object .
                FILTER({s_f} || {o_f})
            }}
        }}
        """

    elif qt == "subject_or_predicate":
        query = f"""
        SELECT ?graph ?subject ?predicate ?object WHERE {{
            GRAPH ?graph {{
                ?subject ?predicate ?object .
                FILTER({s_f} || {p_f})
            }}
        }}
        """

    elif qt == "predicate_or_object":
        query = f"""
        SELECT ?graph ?subject ?predicate ?object WHERE {{
            GRAPH ?graph {{
                ?subject ?predicate ?object .
                FILTER({p_f} || {o_f})
            }}
        }}
        """

    elif qt == "1hop_fixed":
        pred_values = " ".join(f"<urn:placeholder:{p}>" for p in synonyms["predicate"])
        s_f2 = contains_filter("subject", synonyms["subject"])
        o_f2 = contains_filter("object",  synonyms["object"])
        query = f"""
        SELECT ?graph ?subject ?predicate ?object WHERE {{
            GRAPH ?graph {{
                ?subject ?predicate ?object .
                VALUES ?predicate {{ {pred_values} }}
                FILTER({s_f2})
                FILTER({o_f2})
            }}
        }}
        """

    elif qt == "1hop_variable":
        s_f2 = contains_filter("subject",   synonyms["subject"])
        p_f2 = contains_filter("predicate", synonyms["predicate"])
        o_f2 = contains_filter("object",    synonyms["object"])
        query = f"""
        SELECT ?graph ?subject ?predicate ?object WHERE {{
            GRAPH ?graph {{
                ?subject ?predicate ?object .
                FILTER({s_f2} && {p_f2} && {o_f2})
            }}
        }}
        """

    elif qt == "1hop_optional":
        pred_values = " ".join(f"<urn:placeholder:{p}>" for p in synonyms["predicate"])
        s_f2 = contains_filter("subject", synonyms["subject"])
        o_f2 = contains_filter("object",  synonyms["object"])
        query = f"""
        SELECT ?graph ?subject ?predicate ?object WHERE {{
            GRAPH ?graph {{
                ?subject ?predicate ?object .
                VALUES ?predicate {{ {pred_values} }}
                FILTER({s_f2})
                OPTIONAL {{
                    FILTER({o_f2})
                }}
            }}
        }}
        """

    print(f"[SPARQL] Query built for '{qt}' ({len(query)} chars)")
    print(f"[SPARQL] Query:\n{query}")

    # ------------------------------------------------------------------
    # 4. Execute and parse results into (source, subject, predicate, object) quads
    # ------------------------------------------------------------------
    print(f"[SPARQL] Executing query against eu_graph...")
    t0 = time.perf_counter()

    try:
        results = eu_graph.query(query)
    except Exception as e:
        print(f"[SPARQL] ✗ Query execution failed: {type(e).__name__}: {e}")
        raise

    elapsed = time.perf_counter() - t0
    row_vars = [str(v) for v in results.vars]
    print(f"[SPARQL] ✓ Query executed in {elapsed:.3f}s | "
          f"result vars={row_vars}")

    quads: list[tuple[str, str, str, str]] = []

    for i, row in enumerate(results):
        if "subject" in row_vars:
            source = str(row.graph) if "graph" in row_vars and row.graph else ""
            quad = (source, str(row.subject), str(row.predicate), str(row.object))
            quads.append(quad)

            if i < 3:   # Preview first 3 rows to avoid log spam
                print(f"[SPARQL] Row {i}: source='{source}' | "
                      f"s='{str(row.subject)[:60]}' | "
                      f"p='{str(row.predicate)[:60]}' | "
                      f"o='{str(row.object)[:60]}'")

        elif "entity" in row_vars:
            if row.hop1Pred:
                quad = ("", str(row.entity), str(row.hop1Pred), str(row.intermediate))
                quads.append(quad)
                if i < 3:
                    print(f"[SPARQL] Row {i} hop1: "
                          f"'{str(row.entity)[:60]}' -> "
                          f"'{str(row.hop1Pred)[:60]}' -> "
                          f"'{str(row.intermediate)[:60]}'")

            if getattr(row, "hop2Pred", None) and getattr(row, "target", None):
                quad = ("", str(row.intermediate), str(row.hop2Pred), str(row.target))
                quads.append(quad)
                if i < 3:
                    print(f"[SPARQL] Row {i} hop2: "
                          f"'{str(row.intermediate)[:60]}' -> "
                          f"'{str(row.hop2Pred)[:60]}' -> "
                          f"'{str(row.target)[:60]}'")

    print(f"[SPARQL] ✓ Parsed {len(quads)} quads total")
    return quads

sparql_tool_dict = {
    sparql_query: {
        "type": "function",
        "function": {
            "name": "sparql_query_builder",
            "description": """
                Builds and executes a parameterized SPARQL query on an RDF graph.
                Selects the appropriate query type, then extracts keywords and synonyms from the user query, and assigns them to subject, predicate, and object slots.
                Returns results as a list of (source, subject, predicate, object) quads.
                """,
            "parameters": {
                "type": "object",
                "properties": {
                    "synonyms": {
                        "type": "object",
                        "description": (
                            "A dict of three lists of synonyms, one per triple component. "
                            "For 2-hop query types, the predicate list is order-sensitive: "
                            "the first half is assigned to hop 1, the second half to hop 2."
                        ),
                        "properties": {
                            "subject": {
                                "type": "array",
                                "items": {"type": "string"},
                                "description": "Synonyms for the subject slot."
                            },
                            "predicate": {
                                "type": "array",
                                "items": {"type": "string"},
                                "description": "Synonyms for the predicate slot."
                            },
                            "object": {
                                "type": "array",
                                "items": {"type": "string"},
                                "description": "Synonyms for the object slot."
                            }
                        },
                        "required": ["subject", "predicate", "object"]
                    },
                    "query_type": {
                        "type": "string",
                        "description": "The SPARQL query template to use.",
                        "enum": [
                            "and", "or", "not", "and_or_not",
                            "subject_only", "predicate_only", "object_only",
                            "subject_and_object", "subject_and_predicate", "predicate_and_object",
                            "subject_or_object", "subject_or_predicate", "predicate_or_object",
                            "2hop_fixed", "2hop_variable", "2hop_optional"
                        ]
                    }
                },
                "required": ["synonyms", "query_type"]
            }
        }
    }
}
def call_sparql_subagents(queries):
    """Receives a list of queries, and instantiates sparql sub-agents to handle each query using SPARQL."""
    results = list()
    
    def sub_agent_thread(query):
        sub_agent = Legal_Agent(
                system_prompt=sparql_agent_prompt,
                api_key=openai.api_key,
                tool_table=sparql_tool_dict
            )
        return sub_agent.respond_to(query)

    with ThreadPoolExecutor() as executor:
        results = list(executor.map(sub_agent_thread, queries))

    return results

sparql_batch_tool_dict = {
    call_sparql_subagents: {
        "type": "function",
        "function": {
            "name": "call_sparql_subagents",
            "description": "Receives a list of queries, and instantiates sparql sub-agents to handle each query using SPARQL.",
            "parameters": {
                "type": "object",
                "properties": {
                    "queries": {
                        "type": "array",
                        "items": {
                            "type": "string"
                            },
                        "description": "A list of natural language queries to be converted into SPARQL."
                    }
                },
                "required": ["queries"]
            }
        }
    }
}

In [13]:
def describe_graph():
    """Returns classes, properties, and named graphs present in the RDF graph."""
    classes_query = """
        SELECT DISTINCT ?type (COUNT(?s) AS ?n)
        WHERE { ?s rdf:type ?type }
        GROUP BY ?type ORDER BY DESC(?n) LIMIT 30
    """
    props_query = """
        SELECT DISTINCT ?prop (COUNT(*) AS ?n)
        WHERE { ?s ?prop ?o }
        GROUP BY ?prop ORDER BY DESC(?n) LIMIT 50
    """
    graphs_query = "SELECT DISTINCT ?g WHERE { GRAPH ?g { ?s ?p ?o } }"
    return {
        "classes": [row for row in eu_graph.query(classes_query)],
        "properties": [row for row in eu_graph.query(props_query)],
        "named_graphs": [row for row in eu_graph.query(graphs_query)]
    }

#### *SPARQL Sub-Agent Testing:*

In [14]:
sparql_agent = Legal_Agent(
    system_prompt=sparql_agent_prompt,
    step_prompts=sparql_step_prompts,
    api_key=openai.api_key,
    tool_table=sparql_tool_dict
    )

[INIT] Legal_Agent initialized.
[INIT] Tools registered: ['sparql_query_builder']
[INIT] Step prompts loaded: 2


In [15]:
sparql_agent.reset_context()

[RESET] Clearing context only (reasoning log preserved).


In [16]:
sparql_agent.context

[{'role': 'system',
  'content': '\nYou are a sub-agent that performs SPARQL query operations.\nYou are required to do the following:\n\n1.  Choose an appropriate query template that best matches the user query;\n    Extract synonyms from keywords in the provided user query.\n2.  Construct a statement for each quad (source document + triple) returned from the results.\n\nYou will be prompted to do each of these tasks, one at a time.\nDO NOT answer the user directly or interpret results beyond verbalization.\n'}]

In [17]:
result = sparql_agent.sequential_reasoning("What information is there on engine regulations?")

[SEQ] Starting sequential_reasoning | max_depth=10
[SEQ] Query: What information is there on engine regulations?
[COND] Step 1 | depth=0/10
[COND] Formatted prompt (1558 chars): 
    1.  DETERMINE ONE query type from the list below that matches the logic of the original query:
        ["and", "or", "not", "and_or_not", "subjec...
[RESPOND] Appending user message (1558 chars)
[RESPOND] Sending request to model (context length: 2 messages)...
[RESPOND] Model responded in 34.84s | finish_reason=tool_calls | usage=CompletionUsage(completion_tokens=2647, prompt_tokens=857, total_tokens=3504, completion_tokens_details=None, prompt_tokens_details=None)
[REASONING] We need to parse the user request. The user wants us to determine a query type from the list that matches the logic of the original query: "What information is there on engine regulations?" Then compi...
[TOOL] Calling 'sparql_query_builder' with args: {'synonyms': {'subject': ['engin', 'regulat'], 'predicate': [], 'object': ['engin

KeyboardInterrupt: 

In [ ]:
result

{'statements': [],
 'note': 'I need the original user query (or the SPARQL query results) to construct natural‑language statements for each source‑subject‑predicate‑object quad. Please provide the query or the result set.'}

In [18]:
sparql_results = json.loads(result)

print("User Query:", sparql_results["user_query"])
print("SPARQL Query:", sparql_results["sparql_query"])

for res in sparql_results["raw_results"]:
    for part in res:
        print(part)
    print()

TypeError: the JSON object must be str, bytes or bytearray, not dict

In [12]:
print(sparql_agent.context[-1]["content"])

{
  "user_query": "What information is there on engine regulations?",
  "sparql_query": "SELECT ?s ?p ?o WHERE { ?s ?p ?o . FILTER( regex(str(?s), \"engin|motor|regulat|legislat|act\", \"i\") || regex(str(?p), \"engin|motor|regulat|legislat|act\", \"i\") || regex(str(?o), \"engin|motor|regulat|legislat|act\", \"i\") ) } LIMIT 100",
  "keywords": {
    "extracted": {
      "subject": "regulations",
      "predicate": "",
      "object": ""
    },
    "expanded": {
      "subject": "regulat|legislat|act",
      "predicate": "",
      "object": ""
    }
  },
  "raw_results": [],
  "verbalized_results": [],
  "provenance": [],
  "error": "No results found for the query."
}


#### Agent Instantiation:

In [18]:
receptionist = Legal_Agent(system_prompt=receptionist_prompt, step_prompts=receptionist_step_prompts, api_key=openai.api_key)

legal_expert_lookup = dict()
for expert in legal_expert_list:
    expert_instance = Legal_Agent(
        system_prompt=expert["prompt"],
        step_prompts=expert["step_prompts"],
        tool_table=sparql_batch_tool_dict,
        api_key=expert["api_key"]
        )
    legal_expert_lookup[expert["practice_area"]] = expert_instance

evaluator = Legal_Agent(system_prompt=evaluator_prompt, api_key=openai.api_key)

#### *Receptionist Agent Testing:*

In [80]:
receptionist = Legal_Agent(system_prompt=receptionist_prompt, step_prompts=receptionist_step_prompts, api_key=openai.api_key)

In [62]:
receptionist.reset_context()

In [22]:
receptionist.context

[{'role': 'system',
  'content': '\nYou are a front-facing receptionist that handles inquiries about EU law.\nYou are required to do the following:\n\n1.  Reformulate and improve queries;\n2.  Identify ALL practice areas RELEVANT to the reformulated query;\n3.  Respond to queries accordingly.\n\nYou will be prompted to do each of these tasks, one at a time.\n'},
 {'role': 'user',
  'content': '\n    1.  Reformulate the user\'s query via decomposition and/or adding sufficient context.\n        Ensure that queries/sub-queries are atomic to facilitate downstream SPARQL queries.\n        Collect the resulting query/sub-queries in a list.\n        Determine if the user\'s query is general or complex.\n        If the query is general, proceed to step 3 after the outputting the JSON dictionary.\n\n        Output the following JSON dictionary:\n        {\n            "improved_query": [reformulated_query_list],\n            "next_step": 2 or 3 depending on step 1\n        }\n\n        User que

In [26]:
test_queries = json.loads(receptionist.context[2]["content"])["improved_query"]
test_queries

['What consumer rights does a buyer have under EU and German law for a faulty laptop purchased in Berlin?',
 'What obligations does a retailer have in Germany regarding warranty claims for a product that fails during the warranty period?',
 'What steps should a consumer take when a retailer refuses to honor a warranty for a faulty product in Germany?',
 'What legal remedies are available to a consumer in Germany if a retailer refuses to repair, replace, or refund a faulty laptop under warranty?',
 'What are the time limits for making a warranty claim under German law?',
 'What alternative dispute resolution mechanisms are available for consumer disputes in Germany, such as ADR or small claims court?']

In [81]:
result = receptionist.sequential_reasoning(query="Hi there, I'm Jim. I paid for a laptop at Harvey Norman's in Berlin. It broke down while the warranty was still active. but the store doesn't want to claim responsibility. What should I do? ")

In [85]:
print("Reformulated List of Queries:")
for q in result["improved_query"]:
    print("\t", q)
print()
print("Response:", result["response"])
print("Practice Areas:", result["practice_areas"])

Reformulated List of Queries:
	 What are the consumer rights under EU law for non‑conforming goods purchased in Germany?
	 What obligations does a seller have regarding warranty claims for a laptop bought in Berlin?
	 What steps can a consumer take if the seller refuses to accept responsibility for a warranty claim?
	 What remedies (repair, replacement, refund) are available to a consumer under EU law when a product fails during the warranty period?
	 How can a consumer enforce these rights, including possible legal actions, alternative dispute resolution, or small‑claims procedures in Germany?

Response: We have identified the relevant practice areas for your inquiry (Consumer law, EU law, Commercial law, Litigation/mediation/arbitration, and damage to goods). I will forward your questions to the appropriate experts in these fields so they can provide you with detailed guidance on your rights, the seller’s obligations, possible remedies, and the steps you can take to enforce those rig

In [86]:
result = receptionist.sequential_reasoning("Hi there, what is EU law?")

In [87]:
print("Reformulated List of Queries:")
for q in result["improved_query"]:
    print("\t", q)
print()
print("Response:", result["response"])
print("Practice Areas:", result["practice_areas"])

Reformulated List of Queries:
	 What is the definition of EU law?
	 What are the primary sources of EU law (treaties, regulations, directives, decisions, case law)?
	 What are the main policy areas covered by EU law (e.g., internal market, competition, consumer protection, environment, etc.)?
	 How does EU law interact with the national laws of member states (principle of supremacy, direct effect)?
	 Which EU institutions are responsible for creating, adopting, and interpreting EU law (European Commission, European Parliament, Council of the EU, Court of Justice of the EU)?

Response: EU law is the body of legal rules that apply across the European Union, created by its institutions and binding on both member states and individuals. 

The primary sources of EU law are:
1. **Treaties** – the founding treaties (Treaty on European Union and Treaty on the Functioning of the EU) set out the Union’s objectives and powers.
2. **Regulations** – directly applicable in all member states without 

In [27]:
str(test_queries)

"['What consumer rights does a buyer have under EU and German law for a faulty laptop purchased in Berlin?', 'What obligations does a retailer have in Germany regarding warranty claims for a product that fails during the warranty period?', 'What steps should a consumer take when a retailer refuses to honor a warranty for a faulty product in Germany?', 'What legal remedies are available to a consumer in Germany if a retailer refuses to repair, replace, or refund a faulty laptop under warranty?', 'What are the time limits for making a warranty claim under German law?', 'What alternative dispute resolution mechanisms are available for consumer disputes in Germany, such as ADR or small claims court?']"

#### *Legal Expert Agent Testing:*

In [29]:
test_expert = legal_expert_lookup["Consumer law"]

In [31]:
test_expert.reset_context()

In [30]:
test_expert.context

[{'role': 'system',
  'content': '\n    You are a EU legal researcher in the following practice area: Consumer law.\n    You are required to analyze the given queries/sub-queries STRICTLY WITHIN your practice area.\n    You will be tasked to do the following:\n\n    1.  For each query/sub-query, call a SPARQL sub-agent via the sparql_agent tool.\n    2.  Use the IRAC (Issue-Rules-Application-Conclusion) framework to analyze the results.\n    3.  Organize the results into a coherent argument.\n\n    You will be prompted to do each of these tasks, one at a time.\n    '},
 {'role': 'user',
  'content': '\n        1.  For each query/sub-query, call a SPARQL sub-agent via the sparql_agent tool.\n            ONLY SEARCH FOR INFORMATION WITHIN YOUR PRACTICE AREA.\n\n            Output the following JSON dictionary:\n            {\n                "compiled_results": [list of passed statements/propositions],\n                "queries_to_recall": [keep this blank, it\'s for Step 2],\n          

In [32]:
legal_expert_lookup["Consumer law"].sequential_reasoning(str(test_queries))

Step 1


{'path': 'tools/sparql_agent.py', 'line_start': 1, 'line_end': 400}

#### *Evaluator Agent Testing:*

### **Complete Workflow**

In [23]:
def analyze_query(query):
    # Screening by the Receptionist Agent
    screened_results = receptionist.sequential_reasoning(query)

    print("Finished screening query")

    receptionist_response = screened_results.get("response")
    improved_queries = screened_results.get("improved_queries")
    relev_areas = screened_results.get("practice_areas")

    print("Finished parsing receptionist's response")

    # If the query is simple:
    if not relev_areas:
        print("Proceeding to general response")
        return receptionist_response

    # Otherwise, route the reformulated query(ies) to the relevant Legal Experts:
    else:
        # Route queries to relevant Legal Expert Agents
        print("Calling experts:")

        called_experts = [legal_expert_lookup[area] for area in relev_areas]

        expert_analyses = list()

        def expert_thread(expert_agent):
            print(expert_agent.context[-1])
            return expert_agent.sequential_reasoning(improved_queries)
        
        print("Starting batch calls:")

        with ThreadPoolExecutor() as executor:
            expert_analyses = list(executor.map(expert_thread, called_experts))

        # Output format of a Legal Expert Agent
        #   {
        #       "practice_area": {area},
        #       "supporting_propositions": [list of supporting statements/propositions],
        #       "refuting_propositions": [list of refuting statements/propositions]
        #   }

        formatted_analyses = list()
        for analysis in expert_analyses:
            area = analysis.get("practice_area")
            support_props = analysis.get("supporting_propositions")
            refute_props = analysis.get("refuting_propositions")

            support_props_indexed = [f"{idx + 1}.\t{prop}" for idx, prop in enumerate(support_props)]
            refute_props_indexed = [f"{idx + 1}.\t{prop}" for idx, prop in enumerate(refute_props)]

            support_props_concat = "\n".join(support_props_indexed)
            refute_props_concat = "\n".join(refute_props_indexed)

            pretty_analysis = f"""
            Practice Area:\t{area}

            Supporting Propositions:
            {support_props_concat}

            Refuting Propositions:
            {refute_props_concat}
            -------------------------
            """

            formatted_analyses.append(pretty_analysis)

        concatenated_analyses = "\n".join(formatted_analyses)

        print("Passing analyses to evaluator:")

        final_result = json.loads(evaluator.respond_to(concatenated_analyses))

        # Output format of the Evaluator Agent
        #   {
        #       "evaluation": "a comprehensive evaluation",
        #       "supporting_props": [list of supporting statements/propositions],
        #       "refuting_props": [list of refuting statements/propositions],
        #       "inconsistencies": [list of conflicting/inconsistent rules],
        #       "recs_and strats": [list of recommendations and strategies],
        #       "likely_outcomes": [most likely outcomes, if applicable]
        #   }
        
        final_eval = final_result.get("evaluation")
        supporting_props = final_result.get("supporting_props")
        refuting_props = final_result.get("refuting_props")
        inconsistencies = final_result.get("inconsistencies")
        recs_and_strats = final_result.get("recs_and_strats")
        likely_outcomes = final_result.get("likely_outcomes")

        supporting_props_indexed = [f"{idx + 1}.\t{prop}" for idx, prop in enumerate(supporting_props)]
        refuting_props_indexed = [f"{idx + 1}.\t{prop}" for idx, prop in enumerate(refuting_props)]
        inconsistencies_indexed = [f"{idx + 1}.\t{inconsistency}" for idx, inconsistency in enumerate(inconsistencies)]
        recs_and_strats_indexed = [f"{idx + 1}.\t{rec}" for idx, rec in enumerate(recs_and_strats)]
        likely_outcomes_indexed = [f"{idx + 1}.\t{outcome}" for idx, outcome in enumerate(likely_outcomes)]

        supporting_props_concat = "\n".join(supporting_props_indexed)
        refuting_props_concat = "\n".join(refuting_props_indexed)
        inconsistencies_concat = "\n".join(inconsistencies_indexed)
        recs_and_strats_concat = "\n".join(recs_and_strats_indexed)
        likely_outcomes_concat = "\n".join(likely_outcomes_indexed)

        final_response = f"""
        Evaluation:
        {final_eval}

        Supporting Propositions:
        {supporting_props_concat}

        Refuting Propositions:
        {refuting_props_concat}

        Inconsistencies/Conflicting Rules:
        {inconsistencies_concat}

        Recommendations and Strategies:
        {recs_and_strats_concat}
        """

        if likely_outcomes_concat:
            final_response += "\n"
            final_response += f"""
            Likely Outcomes:
            {likely_outcomes_concat}
            """

        return final_response

In [10]:
query = """
Hi there, I'm Jim.
I paid for a laptop at Harvey Norman's in Berlin.
It broke down while the warranty was still active, but the store doesn't want to claim responsibility.
What should I do?
"""

analyze_query(query)

Finished screening query
Finished parsing receptionist's response
Proceeding to general response


'Thank you for your detailed questions regarding your warranty dispute. I will forward your query to the relevant experts in Commercial law, Consumer law, EU law, Litigation, mediation, arbitration, and damage to goods. They will review your situation and provide you with comprehensive guidance on your rights, obligations, remedies, and the steps you can take.'

In [24]:
complex_query = "How do the regulatory objectives, implementation mechanisms, and legal effects" \
                "differ between early Common Agricultural Policy instruments—specifically Regulation No 18/63 and Regulation No 19/65" \
                "—and later harmonization measures such as Directive 66/402/EEC, "\
                "when accounting for interpretative clarifications introduced via corrigenda "\
                "(e.g., Corrigendum to 1963 measure) and implementing decisions like Decision of 30 April 1962? "\
                "In particular, how do these instruments collectively shape market organization, "\
                "seed certification standards, and Member State obligations, "\
                "and what tensions or complementarities emerge between directly applicable regulations and transposition-dependent directives?"

In [25]:
receptionist.context

[{'role': 'system',
  'content': '\nYou are a front-facing receptionist that handles inquiries about EU law.\nYou are required to do the following:\n\n1.  Reformulate and improve queries;\n2.  Identify ALL practice areas RELEVANT to the reformulated query;\n3.  Respond to queries accordingly.\n\nYou will be prompted to do each of these tasks, one at a time.\n'}]

In [ ]:
receptionist.reset_working_memory()
for expert in legal_expert_lookup.values():
    expert.reset_working_memory()

analyze_query(complex_query)

Finished screening query
Finished parsing receptionist's response
Calling experts:
Starting batch calls:
{'role': 'system', 'content': '\n    You are a EU legal researcher in the following practice area: EU law.\n    You are required to analyze the given queries/sub-queries STRICTLY WITHIN your practice area.\n    You will be tasked to do the following:\n\n    1.  For each query/sub-query, call a SPARQL sub-agent via the call_sparql_subagents tool.\n    2.  Use the IRAC (Issue-Rules-Application-Conclusion) framework to analyze the results.\n    3.  Organize the results into a coherent argument.\n\n    You will be prompted to do each of these tasks, one at a time.\n    '}
{'role': 'system', 'content': '\n    You are a EU legal researcher in the following practice area: Public law.\n    You are required to analyze the given queries/sub-queries STRICTLY WITHIN your practice area.\n    You will be tasked to do the following:\n\n    1.  For each query/sub-query, call a SPARQL sub-agent via 

In [8]:
legal_expert_lookup["Human rights and civil liberties"].context

NameError: name 'legal_expert_lookup' is not defined

In [9]:
json.loads(receptionist.context[-1]["content"]).get("practice_areas")

NameError: name 'receptionist' is not defined